# 🌧️ Predicción de Lluvia en Australia con Redes Neuronales

## 🎯 Objetivo del Ejercicio

Desarrollar una red neuronal que prediga si **lloverá mañana** basándose en datos meteorológicos actuales de diferentes ciudades australianas.

---

## 📊 Información del Dataset

**Rain in Australia Dataset:**
- **Fuente**: Observaciones meteorológicas diarias de estaciones australianas (10 años)
- **Muestras**: ~145,460 observaciones
- **Features**: 23 características meteorológicas
- **Target**: `RainTomorrow` (Sí/No - ¿lloverá mañana?)

### 🌡️ Variables Principales:
- **Ubicación**: Ciudad/estación meteorológica
- **Temperatura**: Min, Max, 9am, 3pm
- **Humedad**: 9am, 3pm
- **Presión**: 9am, 3pm
- **Viento**: Velocidad y dirección (9am, 3pm)
- **Nubosidad**: 9am, 3pm
- **Lluvia**: Cantidad hoy, si llovió hoy

### 🎓 Habilidades a Practicar:
1. ✅ Manejo de **valores faltantes** (dataset real con muchos NaN)
2. ✅ **Encoding** de variables categóricas (Location, WindDir)
3. ✅ **Normalización** de features numéricas
4. ✅ Manejo de **desbalanceo** de clases
5. ✅ Construcción y entrenamiento de **redes neuronales**
6. ✅ Evaluación con **métricas apropiadas** (no solo accuracy)
7. ✅ Visualización de resultados

---

## 1️⃣ Instalación de Librerías

In [ ]:
# Instalamos las librerías necesarias
%pip install tensorflow pandas numpy matplotlib seaborn scikit-learn

## 2️⃣ Importación de Librerías

In [ ]:
# Librerías principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    roc_auc_score, 
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
# Configuración
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# Estilo de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"✅ TensorFlow versión: {tf.__version__}")
print(f"✅ Pandas versión: {pd.__version__}")
print(f"✅ GPU disponible: {'Sí' if len(tf.config.list_physical_devices('GPU')) > 0 else 'No (usando CPU)'}")

## 3️⃣ Carga del Dataset

Cargamos el dataset directamente desde Kaggle (también disponible en otras fuentes).

In [ ]:
# URL del dataset
url = 'weatherAUS.csv'

# Cargamos el dataset
print("📥 Descargando dataset... (puede tardar unos segundos)")
df = pd.read_csv(url)

print(f"\n✅ Dataset cargado exitosamente")
print(f"📊 Forma del dataset: {df.shape}")
print(f"📈 Columnas: {df.shape[1]}")
print(f"📈 Filas: {df.shape[0]:,}")

## 4️⃣ Exploración Inicial de Datos (EDA)

In [ ]:
# Primeras filas
print("🔍 Primeras 5 filas del dataset:\n")
display(df.head())

In [ ]:
# Información general
print("📋 Información del dataset:\n")
df.info()

In [ ]:
# Estadísticas descriptivas
print("📊 Estadísticas descriptivas:\n")
display(df.describe())

In [ ]:
# Listado de columnas con sus tipos
print("\n📝 Columnas del dataset:\n")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col:20s} - Tipo: {df[col].dtype}")

### 🎯 Distribución de la Variable Objetivo

In [ ]:
# Distribución de RainTomorrow
print("🎯 Distribución de la variable objetivo (RainTomorrow):\n")
print(df['RainTomorrow'].value_counts())
print("\nPorcentajes:")
print(df['RainTomorrow'].value_counts(normalize=True) * 100)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
df['RainTomorrow'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'coral'], edgecolor='black')
axes[0].set_title('Distribución de RainTomorrow', fontsize=14, fontweight='bold')
axes[0].set_xlabel('¿Llueve mañana?', fontsize=12)
axes[0].set_ylabel('Cantidad', fontsize=12)
axes[0].set_xticklabels(['No', 'Sí'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Gráfico de pastel
df['RainTomorrow'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                        colors=['skyblue', 'coral'], startangle=90)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Análisis de desbalanceo
no_rain = (df['RainTomorrow'] == 'No').sum()
yes_rain = (df['RainTomorrow'] == 'Yes').sum()
ratio = no_rain / yes_rain if yes_rain > 0 else 0

print(f"\n⚖️ Ratio de desbalanceo: {ratio:.2f}:1 (No lluvia : Sí lluvia)")
if ratio > 2:
    print("⚠️ El dataset está DESBALANCEADO - Consideraremos técnicas de balanceo")

### 📍 Análisis de Valores Faltantes

In [ ]:
# Conteo de valores faltantes
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores Faltantes': missing.values,
    'Porcentaje (%)': missing_percent.values
}).sort_values('Valores Faltantes', ascending=False)

print("❌ Valores faltantes por columna:\n")
print(missing_df[missing_df['Valores Faltantes'] > 0])

# Visualización
plt.figure(figsize=(12, 6))
missing_data = missing_df[missing_df['Valores Faltantes'] > 0].sort_values('Porcentaje (%)', ascending=True)
plt.barh(missing_data['Columna'], missing_data['Porcentaje (%)'], color='salmon', edgecolor='black')
plt.xlabel('Porcentaje de Valores Faltantes (%)', fontsize=12)
plt.title('Valores Faltantes por Columna', fontsize=14, fontweight='bold')
plt.axvline(x=40, color='red', linestyle='--', label='40% umbral')
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 🗺️ Análisis por Ubicación

In [ ]:
# Top 10 ubicaciones con más registros
top_locations = df['Location'].value_counts().head(10)

print("📍 Top 10 Ubicaciones con más observaciones:\n")
print(top_locations)

# Visualización
plt.figure(figsize=(12, 5))
top_locations.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 10 Ubicaciones con Más Observaciones', fontsize=14, fontweight='bold')
plt.xlabel('Ubicación', fontsize=12)
plt.ylabel('Cantidad de Observaciones', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5️⃣ Preprocesamiento de Datos

### 5.1 Eliminar columnas innecesarias

In [ ]:
# Eliminamos la columna Date (no aporta información predictiva directa)
# Podríamos extraer features como mes, día, etc., pero lo simplificaremos
print("🗑️ Eliminando columna 'Date'...")
df_clean = df.drop(['Date'], axis=1)

print(f"✅ Nueva forma del dataset: {df_clean.shape}")

### 5.2 Eliminar filas con target faltante

In [ ]:
# No podemos entrenar sin la variable objetivo
print(f"📊 Filas antes de eliminar NaN en target: {len(df_clean):,}")
df_clean = df_clean.dropna(subset=['RainTomorrow'])
print(f"📊 Filas después de eliminar NaN en target: {len(df_clean):,}")
print(f"🗑️ Eliminadas: {len(df) - len(df_clean):,} filas")

### 5.3 Separar Features y Target

In [ ]:
# Separamos X (features) e y (target)
X = df_clean.drop('RainTomorrow', axis=1)
y = df_clean['RainTomorrow']

print(f"✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")

### 5.4 Identificar Columnas Numéricas y Categóricas

In [ ]:
# Identificamos tipos de columnas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"🔢 Features numéricas ({len(numeric_features)}):")
print(numeric_features)
print(f"\n📝 Features categóricas ({len(categorical_features)}):")
print(categorical_features)

### 5.5 Imputación de Valores Faltantes

**Estrategia:**
- **Numéricas**: Rellenar con la mediana (más robusta a outliers)
- **Categóricas**: Rellenar con la moda (valor más frecuente)

In [ ]:
# Imputación de valores faltantes en numéricas (mediana)
print("🔧 Imputando valores faltantes en features numéricas...")
for col in numeric_features:
    if X[col].isnull().sum() > 0:
        median_val = X[col].median()
        print(f"   ✓ {col}: {X[col].isnull().sum()} faltantes → rellenados con mediana ({median_val:.2f})")
        X[col].fillna(median_val, inplace=True)
        print(f"   ✓ {col}: {X[col].isnull().sum()} faltantes → rellenados con mediana ({median_val:.2f})")

# Imputación de valores faltantes en categóricas (moda)
print("\n🔧 Imputando valores faltantes en features categóricas...")
for col in categorical_features:
    if X[col].isnull().sum() > 0:
        mode_val = X[col].mode()[0]
        X[col].fillna(mode_val, inplace=True)
        print(f"   ✓ {col}: {X[col].isnull().sum()} faltantes → rellenados con moda ('{mode_val}')")

# Verificamos
print(f"\n✅ Total de valores faltantes restantes: {X.isnull().sum().sum()}")

### 5.6 Encoding de Variables Categóricas

Convertimos las variables categóricas a numéricas usando **Label Encoding** para variables binarias y **One-Hot Encoding** para variables con múltiples categorías.

In [ ]:
# Label Encoding para variables binarias (Yes/No)
binary_features = ['RainToday']

print("🔤 Aplicando Label Encoding a variables binarias...")
le = LabelEncoder()
for col in binary_features:
    if col in X.columns:
        X[col] = le.fit_transform(X[col])
        print(f"   ✓ {col}: convertido a 0/1")

# One-Hot Encoding para variables con múltiples categorías
print("\n🔤 Aplicando One-Hot Encoding a variables categóricas...")
multi_categorical = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']

X_encoded = pd.get_dummies(X, columns=multi_categorical, drop_first=True)

print(f"\n✅ Shape después de encoding: {X_encoded.shape}")
print(f"📊 Nuevas features creadas: {X_encoded.shape[1] - X.shape[1]}")

### 5.7 Encoding de la Variable Objetivo

In [ ]:
# Convertimos RainTomorrow a 0/1
y_encoded = (y == 'Yes').astype(int)

print("🎯 Variable objetivo codificada:")
print(f"   No (0): {(y_encoded == 0).sum():,}")
print(f"   Sí (1): {(y_encoded == 1).sum():,}")

### 5.8 División Train/Test

In [ ]:
# Dividimos en train (80%) y test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, 
    y_encoded, 
    test_size=0.2, 
    random_state=42,
    stratify=y_encoded  # Mantiene la proporción de clases
)

print("📂 División de datos:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test:  {y_test.shape}")

print(f"\n✅ Proporción train/test: {len(X_train)/(len(X_train)+len(X_test))*100:.1f}% / {len(X_test)/(len(X_train)+len(X_test))*100:.1f}%")

### 5.9 Normalización de Features

Escalamos las features al rango [0, 1] o con media 0 y desviación 1 para que la red neuronal converja mejor.

In [ ]:
# Normalizamos usando StandardScaler
scaler = StandardScaler()

print("⚙️ Normalizando features...")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Features normalizadas")
print(f"   Media de X_train: {X_train_scaled.mean():.6f}")
print(f"   Std de X_train: {X_train_scaled.std():.6f}")

## 6️⃣ Construcción del Modelo de Red Neuronal

### Arquitectura de la Red:

```
Entrada → Capa Oculta 1 (128 neuronas, ReLU) → Dropout (30%)
       → Capa Oculta 2 (64 neuronas, ReLU)  → Dropout (30%)
       → Capa Oculta 3 (32 neuronas, ReLU)  → Dropout (20%)
       → Salida (1 neurona, Sigmoid)
```

**Decisiones de diseño:**
- **3 capas ocultas**: Suficiente capacidad para aprender patrones complejos
- **ReLU**: Activación rápida y efectiva
- **Dropout alto (30%)**: Previene overfitting en dataset grande
- **Sigmoid en salida**: Para clasificación binaria (probabilidad 0-1)
- **Binary Crossentropy**: Función de pérdida para clasificación binaria

In [ ]:
# Creamos el modelo
def crear_modelo(input_dim):
    model = models.Sequential([
        # Capa de entrada
        layers.Input(shape=(input_dim,)),
        
        # Primera capa oculta
        layers.Dense(128, activation='relu', name='capa_oculta_1'),
        layers.Dropout(0.3, name='dropout_1'),
        
        # Segunda capa oculta
        layers.Dense(64, activation='relu', name='capa_oculta_2'),
        layers.Dropout(0.3, name='dropout_2'),
        
        # Tercera capa oculta
        layers.Dense(32, activation='relu', name='capa_oculta_3'),
        layers.Dropout(0.2, name='dropout_3'),
        
        # Capa de salida
        layers.Dense(1, activation='sigmoid', name='salida')
    ])
    
    return model

# Creamos el modelo
model = crear_modelo(X_train_scaled.shape[1])

# Mostramos la arquitectura
print("🏗️ Arquitectura del modelo:\n")
model.summary()

## 7️⃣ Compilación del Modelo

**Configuración:**
- **Optimizador**: Adam (adaptativo, rápido)
- **Pérdida**: Binary Crossentropy (para clasificación binaria)
- **Métricas**: Accuracy, Precision, Recall, AUC

In [ ]:
# Compilamos el modelo con métricas adicionales
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

print("✅ Modelo compilado correctamente")

## 8️⃣ Entrenamiento del Modelo

**Hiperparámetros:**
- **Epochs**: 20 (podemos parar antes si converge)
- **Batch size**: 256 (buen balance entre velocidad y estabilidad)
- **Validation split**: 15% (para monitoreo)
- **Class weights**: Para manejar el desbalanceo de clases

In [ ]:
# Calculamos class weights para manejar el desbalanceo
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array)) # Convertimos a diccionario

print("⚖️ Pesos de clase (para manejar desbalanceo):")
print(f"   Clase 0 (No lluvia): {class_weights[0]:.2f}")
print(f"   Clase 1 (Sí lluvia): {class_weights[1]:.2f}")

In [ ]:
# Entrenamos el modelo
print("🚀 Iniciando entrenamiento...\n")

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=256,
    validation_split=0.15,
    class_weight=class_weights,
    verbose=1
)

print("\n✅ Entrenamiento completado")

### 📊 Visualización del Entrenamiento

In [ ]:
# Graficamos las métricas del entrenamiento
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Entrenamiento', marker='o', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validación', marker='s', linewidth=2)
axes[0, 0].set_title('Evolución de la Pérdida', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Pérdida')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Entrenamiento', marker='o', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validación', marker='s', linewidth=2)
axes[0, 1].set_title('Evolución de la Precisión', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Época')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Entrenamiento', marker='o', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Validación', marker='s', linewidth=2)
axes[1, 0].set_title('Evolución de la Precisión (Precision)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Época')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Entrenamiento', marker='o', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Validación', marker='s', linewidth=2)
axes[1, 1].set_title('Evolución del Recall', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Época')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9️⃣ Evaluación del Modelo

In [ ]:
# Evaluamos en el conjunto de test
print("📊 Evaluando modelo en conjunto de prueba...\n")

test_results = model.evaluate(X_test_scaled, y_test, verbose=0)

print("🎯 RESULTADOS EN EL CONJUNTO DE PRUEBA:")
print(f"   Loss:      {test_results[0]:.4f}")
print(f"   Accuracy:  {test_results[1]:.4f} ({test_results[1]*100:.2f}%)")
print(f"   Precision: {test_results[2]:.4f}")
print(f"   Recall:    {test_results[3]:.4f}")
print(f"   AUC:       {test_results[4]:.4f}")

### 🎯 Predicciones y Métricas Detalladas

In [ ]:
# Obtenemos predicciones
y_pred_proba = model.predict(X_test_scaled, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Calculamos métricas adicionales
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("\n📈 MÉTRICAS DETALLADAS:")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f} (De todas las predicciones positivas, {precision*100:.1f}% fueron correctas)")
print(f"   Recall:    {recall:.4f} (De todos los casos positivos reales, detectamos {recall*100:.1f}%)")
print(f"   F1-Score:  {f1:.4f} (Media armónica entre Precision y Recall)")
print(f"   AUC-ROC:   {auc:.4f} (Capacidad de discriminación del modelo)")

### 📊 Matriz de Confusión

In [ ]:
# Calculamos la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

# Visualizamos
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, 
            xticklabels=['No Lluvia', 'Sí Lluvia'],
            yticklabels=['No Lluvia', 'Sí Lluvia'],
            cbar_kws={'label': 'Cantidad'})
plt.title('Matriz de Confusión - Predicción de Lluvia', fontsize=14, fontweight='bold')
plt.ylabel('Valor Real', fontsize=12)
plt.xlabel('Predicción', fontsize=12)

# Añadimos porcentajes
for i in range(2):
    for j in range(2):
        percentage = cm[i, j] / cm.sum() * 100
        plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

# Interpretación
tn, fp, fn, tp = cm.ravel()
print("\n🔍 INTERPRETACIÓN DE LA MATRIZ:")
print(f"   ✅ Verdaderos Negativos (TN): {tn:,} - Predijo correctamente 'No lluvia'")
print(f"   ❌ Falsos Positivos (FP):     {fp:,} - Predijo 'Sí lluvia' pero no llovió")
print(f"   ❌ Falsos Negativos (FN):     {fn:,} - Predijo 'No lluvia' pero sí llovió")
print(f"   ✅ Verdaderos Positivos (TP): {tp:,} - Predijo correctamente 'Sí lluvia'")

### 📈 Curva ROC

In [ ]:
# Calculamos la curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Visualizamos
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Línea base (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
plt.title('Curva ROC - Predicción de Lluvia', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 AUC-ROC: {roc_auc:.4f}")
if roc_auc > 0.9:
    print("   🌟 EXCELENTE discriminación")
elif roc_auc > 0.8:
    print("   ✅ MUY BUENA discriminación")
elif roc_auc > 0.7:
    print("   ✅ BUENA discriminación")
else:
    print("   ⚠️ Discriminación ACEPTABLE")

### 📋 Reporte de Clasificación Completo

In [ ]:
# Reporte detallado
print("\n📋 REPORTE DE CLASIFICACIÓN COMPLETO:\n")
print(classification_report(y_test, y_pred, 
                          target_names=['No Lluvia', 'Sí Lluvia'],
                          digits=4))

## 🔟 Guardar el Modelo

In [ ]:
# Guardamos el modelo
model.save('modelo_rain_australia.keras')
print("💾 Modelo guardado como 'modelo_rain_australia.keras'")

# También guardamos el scaler
import joblib
joblib.dump(scaler, 'scaler_rain_australia.pkl')
print("💾 Scaler guardado como 'scaler_rain_australia.pkl'")

## 1️⃣1️⃣ Función de Predicción para Nuevos Datos

In [ ]:
def predecir_lluvia_manana(datos_hoy, model, scaler):
    """
    Predice si lloverá mañana basándose en datos meteorológicos de hoy.
    
    Args:
        datos_hoy: DataFrame con las features necesarias
        model: Modelo entrenado
        scaler: Scaler ajustado
    
    Returns:
        Predicción y probabilidad
    """
    # Normalizamos
    datos_scaled = scaler.transform(datos_hoy)
    
    # Predicción
    probabilidad = model.predict(datos_scaled, verbose=0)[0][0]
    prediccion = "Sí" if probabilidad > 0.5 else "No"
    
    return prediccion, probabilidad

print("✅ Función 'predecir_lluvia_manana()' definida")

### 🧪 Ejemplo de Predicción

In [ ]:
# Seleccionamos algunos ejemplos aleatorios del conjunto de test
print("🧪 EJEMPLOS DE PREDICCIÓN:\n")

indices = np.random.choice(len(X_test_scaled), 10, replace=False)

for i, idx in enumerate(indices, 1):
    X_ejemplo = X_test_scaled[idx:idx+1]
    y_real = y_test.iloc[idx]
    
    # Predicción
    prob = model.predict(X_ejemplo, verbose=0)[0][0]
    pred = "Sí" if prob > 0.5 else "No"
    real = "Sí" if y_real == 1 else "No"
    
    # Resultado
    emoji = "✅" if pred == real else "❌"
    print(f"{emoji} Ejemplo {i}: Predicción: {pred} (prob: {prob:.2%}) | Real: {real}")

## 1️⃣2️⃣ Análisis de Importancia de Features (Opcional)

Aunque las redes neuronales son "cajas negras", podemos aproximar la importancia de features usando permutaciones.

In [ ]:
# Esta celda es OPCIONAL y puede tardar varios minutos
# Descomenta para ejecutar

# from sklearn.inspection import permutation_importance

# print("🔍 Calculando importancia de features... (puede tardar)\n")

# # Calculamos importancia
# result = permutation_importance(model, X_test_scaled, y_test, 
#                                n_repeats=10, random_state=42, n_jobs=-1)

# # Ordenamos por importancia
# feature_names = X_encoded.columns
# importances = result.importances_mean
# indices = np.argsort(importances)[::-1][:15]  # Top 15

# # Visualizamos
# plt.figure(figsize=(10, 6))
# plt.barh(range(15), importances[indices], color='steelblue', edgecolor='black')
# plt.yticks(range(15), [feature_names[i] for i in indices])
# plt.xlabel('Importancia', fontsize=12)
# plt.title('Top 15 Features Más Importantes', fontsize=14, fontweight='bold')
# plt.gca().invert_yaxis()
# plt.tight_layout()
# plt.show()

## 🔬 **Experimento: Spatial Dropout para Features Correlacionadas**

### ¿Por qué Spatial Dropout?

En nuestro dataset meteorológico, muchas features están **altamente correlacionadas**:

- **Temperaturas**: `MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm`
- **Presiones**: `Pressure9am`, `Pressure3pm`
- **Humedades**: `Humidity9am`, `Humidity3pm`

Estas features tienden a variar juntas (si la temperatura a las 9am es alta, probablemente la de las 3pm también lo sea). El **Dropout regular** apaga neuronas de forma individual, pero cuando las features están correlacionadas, la red puede "compensar" usando otra feature correlacionada.

**Spatial Dropout** apaga **canales completos** en lugar de neuronas individuales, forzando a la red a no depender de grupos de features correlacionadas.

### Arquitectura del Modelo con Spatial Dropout

Vamos a crear un modelo que agrupa las features correlacionadas y aplica Spatial Dropout:

```
                ┌─────────────────────┐
                │   Input (65 dims)   │
                └──────────┬──────────┘
                           │
            ┌──────────────┴──────────────┐
            │   Reshape para 1D Spatial   │
            │   (batch, features, 1)      │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │  SpatialDropout1D (0.3)     │
            │  Apaga canales completos    │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │        Flatten              │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dense(128) + ReLU        │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dropout Regular (0.4)    │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dense(64) + ReLU         │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dropout Regular (0.3)    │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dense(32) + ReLU         │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dropout Regular (0.2)    │
            └──────────────┬──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │    Dense(1) + Sigmoid       │
            │    Output: RainTomorrow     │
            └──────────────────────────────┘
```

### Diferencias clave:

1. **Capa inicial**: `SpatialDropout1D` después del input
2. **Efecto**: Apaga features completas (ej: si apaga `MinTemp`, también fuerza la red a no depender solo de `MaxTemp`)
3. **Capas posteriores**: Dropout regular para evitar overfitting en las capas densas

In [ ]:
%pip install tensorflow pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Importamos tensorflow
import tensorflow as tf
# Importamos las capas necesarias
from tensorflow.keras.layers import SpatialDropout1D, Reshape, Flatten, Dense, Dropout

    
# Instalamos la librería necesaria
from tensorflow.keras.models import Sequential
# 
# Crear el modelo con Spatial Dropout
modelo_spatial = Sequential([
    # Capa de entrada: necesitamos reshape para SpatialDropout1D
    # SpatialDropout1D espera entrada con forma (batch, timesteps, channels)
    # Convertimos (batch, 65) -> (batch, 65, 1)
    Reshape((X_train_scaled.shape[1], 1), input_shape=(X_train_scaled.shape[1],)),
    
    # Spatial Dropout: apaga canales completos (features completas)
    # Tasa del 40% - más conservadora que las capas posteriores
    SpatialDropout1D(0.25),
    
    # Volver a formato plano
    Flatten(),
    
    # Capas densas con Dropout regular
    Dense(128, activation='relu', name='capa_oculta_1_Spatial'),
    Dropout(0.4),
    
    Dense(64, activation='relu', name='capa_oculta_2_Spatial'),
    Dropout(0.3),
    
    Dense(32, activation='relu', name='capa_oculta_3_Spatial'),
    Dropout(0.2),
    
    # Capa de salida
    Dense(1, activation='sigmoid')
])

# Compilar el modelo
modelo_spatial.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

print("=" * 70)
print("MODELO CON SPATIAL DROPOUT")
print("=" * 70)
modelo_spatial.summary()
print("\n💡 Nota: SpatialDropout1D apaga features completas, no neuronas individuales")
print("   Esto es útil cuando las features están correlacionadas (temps, presiones, etc.)")


In [ ]:
# Entrenar el modelo con Spatial Dropout
print("\n🚀 Entrenando modelo con Spatial Dropout...")
print("   (Esto puede tardar varios minutos)")
# Añadimos EarlyStopping para evitar overfitting
from tensorflow.keras.callbacks import EarlyStopping
historia_spatial = modelo_spatial.fit(
    X_train_scaled, 
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    class_weight=class_weights,
    verbose=1,
    callbacks=[
        EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
    ]
)

print("\n✅ Entrenamiento completado")

In [ ]:
# Evaluar el modelo con Spatial Dropout
print("\n" + "=" * 70)
print("EVALUACIÓN DEL MODELO CON SPATIAL DROPOUT")
print("=" * 70)

loss_spatial, accuracy_spatial, precision_spatial, recall_spatial, auc_spatial = modelo_spatial.evaluate(
    X_test_scaled, 
    y_test, 
    verbose=0
)

# Predicciones
y_pred_spatial = (modelo_spatial.predict(X_test_scaled) > 0.5).astype(int)
f1_spatial = f1_score(y_test, y_pred_spatial)

print(f"\n📊 Métricas del modelo con Spatial Dropout:")
print(f"   • Accuracy:  {accuracy_spatial:.4f}")
print(f"   • Precision: {precision_spatial:.4f}")
print(f"   • Recall:    {recall_spatial:.4f}")
print(f"   • F1-Score:  {f1_spatial:.4f}")
print(f"   • AUC-ROC:   {auc_spatial:.4f}")

### 📊 Comparación: Modelo Original vs Spatial Dropout

In [ ]:
# Comparación visual de los dos modelos
import pandas as pd

# Crear DataFrame de comparación (deberás ajustar con las métricas reales del modelo original)
comparacion = pd.DataFrame({
    'Modelo': ['Original (Dropout)', 'Spatial Dropout'],
    'Accuracy': [accuracy, accuracy_spatial],  # Usa las variables del modelo original
    'Precision': [precision, precision_spatial],
    'Recall': [recall, recall_spatial],
    'F1-Score': [f1, f1_spatial],
    'AUC-ROC': [auc, auc_spatial]
})

print("\n" + "=" * 80)
print("COMPARACIÓN DE MODELOS")
print("=" * 80)
print(comparacion.to_string(index=False))
print("=" * 80)

# Calcular diferencias
print("\n📈 Diferencias (Spatial - Original):")
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']:
    diff = comparacion[col].iloc[1] - comparacion[col].iloc[0]
    emoji = "📈" if diff > 0 else "📉" if diff < 0 else "➡️"
    print(f"   {emoji} {col:12s}: {diff:+.4f}")

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Barras comparativas
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x = np.arange(len(metricas))
width = 0.35

valores_original = comparacion.iloc[0, 1:].values
valores_spatial = comparacion.iloc[1, 1:].values

axes[0].bar(x - width/2, valores_original, width, label='Dropout Regular', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, valores_spatial, width, label='Spatial Dropout', color='coral', alpha=0.8)
axes[0].set_xlabel('Métricas', fontsize=12)
axes[0].set_ylabel('Valor', fontsize=12)
axes[0].set_title('Comparación de Métricas: Dropout vs Spatial Dropout', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metricas, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1])

# Gráfico 2: Curvas de aprendizaje comparadas
axes[1].plot(history.history['accuracy'], label='Train Original', color='steelblue', linestyle='--', alpha=0.7)
axes[1].plot(history.history['val_accuracy'], label='Val Original', color='steelblue', linewidth=2)
axes[1].plot(historia_spatial.history['accuracy'], label='Train Spatial', color='coral', linestyle='--', alpha=0.7)
axes[1].plot(historia_spatial.history['val_accuracy'], label='Val Spatial', color='coral', linewidth=2)
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Curvas de Aprendizaje: Comparación', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Interpretación:")
print("   • Si Spatial Dropout mejora Recall: Mejor detección de lluvias (menos falsos negativos)")
print("   • Si mejora Precision: Menos falsas alarmas de lluvia")
print("   • Si las curvas son más suaves: Mejor generalización (menos overfitting)")
print("   • Si AUC aumenta: Mejor discriminación entre las clases en general")

### 🎯 **Conclusiones del Experimento con Spatial Dropout**

#### ¿Cuándo usar Spatial Dropout?

| Situación | Tipo de Dropout Recomendado |
|-----------|------------------------------|
| Features **independientes** (ej: edad, género, categorías) | **Dropout Regular** |
| Features **correlacionadas** (ej: temperaturas, presiones, humedades) | **Spatial Dropout** |
| Datos de **series temporales** o **secuencias** | **Spatial Dropout** |
| Imágenes con **convoluciones** | **Spatial Dropout2D** |
| Capas **densas/fully connected** | **Dropout Regular** |

#### Interpretación de Resultados

**Si Spatial Dropout funciona mejor:**
- ✅ Confirma que las features meteorológicas están correlacionadas
- ✅ El modelo aprende a no depender solo de un grupo de features similares
- ✅ Mejor generalización cuando hay variaciones en los patrones correlacionados

**Si Dropout Regular funciona mejor:**
- ✅ Las features son más independientes de lo esperado
- ✅ O el dataset no tiene suficiente redundancia entre features

**Si ambos dan resultados similares:**
- ✅ El modelo es robusto
- ✅ La regularización está bien balanceada en ambos casos

#### 💡 Lecciones Aprendidas

1. **Spatial Dropout es especializado**: No siempre es mejor, solo cuando hay correlación entre features
2. **Combinar técnicas**: Spatial Dropout al inicio + Dropout regular en capas densas puede ser óptimo
3. **Experimentar es clave**: La única forma de saber qué funciona mejor es probarlo
4. **Monitorear curvas**: Las curvas de validación te dirán si estás reduciendo overfitting

---

**🔬 Para explorar más:**
- Prueba diferentes tasas de dropout (0.2, 0.3, 0.5)
- Combina con regularización L1/L2
- Experimenta con BatchNormalization en lugar de Dropout
- Analiza qué features específicas se correlacionan más usando la matriz de correlación

### 🔍 **Verificación: ¿Cómo comprobar que el Dropout está funcionando?**

#### El Dropout NO pone los pesos a cero

**⚠️ Aclaración importante**: El Dropout **NO modifica los pesos** del modelo. Lo que hace es:

1. Durante el **entrenamiento**: Apaga neuronas aleatoriamente (pone sus **salidas** a 0)
2. Durante la **inferencia/predicción**: Usa todas las neuronas pero escala sus salidas

**Los pesos de las conexiones permanecen intactos** - solo se multiplican las activaciones por 0 durante el entrenamiento.

#### ¿Qué podemos verificar entonces?

1. **Activaciones durante entrenamiento** (salidas de neuronas apagadas = 0)
2. **Comportamiento diferente** entre modo entrenamiento y modo predicción
3. **Reducción de overfitting** en las curvas de aprendizaje

Veamos cómo hacerlo:

In [ ]:
# ============================================================================
# MÉTODO 1: Verificar que los pesos NO son cero (permanecen intactos)
# ============================================================================

print("=" * 80)
print("MÉTODO 1: Inspección de Pesos del Modelo")
print("=" * 80)

# Obtener los pesos de la primera capa densa del modelo original
primera_capa_densa = model.get_layer('capa_oculta_1')  # Primera Dense después del input
pesos, sesgos = primera_capa_densa.get_weights()

print(f"\n📊 Primera capa densa del modelo:")
print(f"   • Forma de los pesos: {pesos.shape}")
print(f"   • Forma de los sesgos: {sesgos.shape}")

# Comprobar que NO hay pesos en cero
pesos_cero = np.sum(pesos == 0)
total_pesos = pesos.size

print(f"\n🔍 Análisis de pesos:")
print(f"   • Total de pesos: {total_pesos:,}")
print(f"   • Pesos exactamente = 0: {pesos_cero}")
print(f"   • Porcentaje de ceros: {(pesos_cero/total_pesos)*100:.2f}%")

print(f"\n✅ Estadísticas de los pesos:")
print(f"   • Media: {pesos.mean():.6f}")
print(f"   • Desviación estándar: {pesos.std():.6f}")
print(f"   • Mínimo: {pesos.min():.6f}")
print(f"   • Máximo: {pesos.max():.6f}")

print("\n💡 Conclusión: Los pesos NO están en cero. El Dropout no modifica los pesos,")
print("   solo las activaciones (salidas) de las neuronas durante el entrenamiento.")

# Visualización de la distribución de pesos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Histograma de pesos
axes[0].hist(pesos.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Cero')
axes[0].set_xlabel('Valor del peso', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].set_title('Distribución de Pesos (Primera Capa Densa)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Gráfico 2: Heatmap de una muestra de pesos
muestra_pesos = pesos[:20, :20]  # Tomar una muestra pequeña para visualizar
im = axes[1].imshow(muestra_pesos, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
axes[1].set_xlabel('Neurona de salida', fontsize=12)
axes[1].set_ylabel('Feature de entrada', fontsize=12)
axes[1].set_title('Mapa de Calor de Pesos (muestra 20x20)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Valor del peso')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# MÉTODO 2: Comparar predicciones con training=True vs training=False
# ============================================================================

print("\n" + "=" * 80)
print("MÉTODO 2: Activación del Dropout - Modo Entrenamiento vs Predicción")
print("=" * 80)

# Tomar una muestra de datos
muestra = X_test_scaled[:5]

print("\n🔬 Probando con 5 muestras de test:")
print(f"   Forma de la muestra: {muestra.shape}")

# Predicción en modo PREDICCIÓN (Dropout desactivado - comportamiento normal)
predicciones_prediccion = model(muestra, training=False).numpy()

# Predicción en modo ENTRENAMIENTO (Dropout activado - neuronas apagadas aleatoriamente)
predicciones_entrenamiento_1 = model(muestra, training=True).numpy()
predicciones_entrenamiento_2 = model(muestra, training=True).numpy()
predicciones_entrenamiento_3 = model(muestra, training=True).numpy()

print("\n" + "=" * 80)
print("RESULTADOS:")
print("=" * 80)

print("\n📊 Predicciones para cada muestra:")
print("-" * 80)
print(f"{'Muestra':<10} {'Predicción':<15} {'Training-1':<15} {'Training-2':<15} {'Training-3':<15}")
print("-" * 80)

for i in range(5):
    print(f"{i+1:<10} {predicciones_prediccion[i,0]:.6f}     "
          f"{predicciones_entrenamiento_1[i,0]:.6f}     "
          f"{predicciones_entrenamiento_2[i,0]:.6f}     "
          f"{predicciones_entrenamiento_3[i,0]:.6f}")

print("-" * 80)

# Calcular varianza en modo entrenamiento
print("\n🔍 Análisis de variabilidad:")
for i in range(5):
    valores_training = [
        predicciones_entrenamiento_1[i,0],
        predicciones_entrenamiento_2[i,0],
        predicciones_entrenamiento_3[i,0]
    ]
    varianza = np.var(valores_training)
    print(f"   Muestra {i+1}: Predicción estable = {predicciones_prediccion[i,0]:.6f}, "
          f"Varianza en training = {varianza:.8f}")

print("\n✅ Interpretación:")
print("   • Modo PREDICCIÓN (training=False): Siempre da el mismo resultado")
print("   • Modo ENTRENAMIENTO (training=True): Resultados varían por Dropout aleatorio")
print("   • Mayor varianza en training = Dropout está apagando neuronas aleatoriamente")

# Visualización
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(5)
width = 0.2

ax.bar(x - 1.5*width, predicciones_prediccion.flatten(), width, 
       label='Predicción (Dropout OFF)', color='green', alpha=0.8)
ax.bar(x - 0.5*width, predicciones_entrenamiento_1.flatten(), width, 
       label='Training pass 1', color='coral', alpha=0.6)
ax.bar(x + 0.5*width, predicciones_entrenamiento_2.flatten(), width, 
       label='Training pass 2', color='coral', alpha=0.6)
ax.bar(x + 1.5*width, predicciones_entrenamiento_3.flatten(), width, 
       label='Training pass 3', color='coral', alpha=0.6)

ax.set_xlabel('Muestra', fontsize=12)
ax.set_ylabel('Probabilidad de lluvia', fontsize=12)
ax.set_title('Dropout: Predicciones Estables vs Variables', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Muestra {i+1}' for i in range(5)])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# MÉTODO 3: Extraer activaciones intermedias y verificar ceros
# ============================================================================

print("\n" + "=" * 80)
print("MÉTODO 3: Inspección de Activaciones Intermedias")
print("=" * 80)

# Mostrar las capas para localizar el Dropout
for idx, layer in enumerate(model.layers):
    print(f"Capa {idx}: {layer.name} ({layer.__class__.__name__})")

# Asumiendo que el primer Dropout está en la capa 2 (ajusta si es necesario)
capa_dropout_idx = 2  # Ajusta según la arquitectura

print(f"\n🔬 Extrayendo activaciones de la capa: (índice {capa_dropout_idx}) "
      f"{model.layers[capa_dropout_idx].name}")

# Tomar una muestra
muestra_activacion = X_test_scaled[:10]

# Asegurarnos de que el modelo esté construido
# Para modelos Sequential, necesitamos construirlo explícitamente
if not model.built:
    model.build(input_shape=(None, X_test_scaled.shape[1]))

# Hacer una llamada al modelo para asegurarnos de que está completamente construido
_ = model(muestra_activacion[:1], training=False)

# Crear un modelo funcional usando las capas existentes
input_layer = tf.keras.Input(shape=(X_test_scaled.shape[1],))
x = input_layer
for i in range(capa_dropout_idx + 1):
    x = model.layers[i](x)

modelo_intermedio = tf.keras.Model(inputs=input_layer, outputs=x)

# Obtener activaciones en modo PREDICCIÓN (Dropout OFF)
activaciones_prediccion = modelo_intermedio(muestra_activacion, training=False).numpy()

# Obtener activaciones en modo ENTRENAMIENTO (Dropout ON)
activaciones_entrenamiento = modelo_intermedio(muestra_activacion, training=True).numpy()

print(f"\n📊 Forma de las activaciones: {activaciones_prediccion.shape}")
print(f"   (10 muestras x {activaciones_prediccion.shape[1]} neuronas)")

# Contar neuronas apagadas (activación = 0)
ceros_prediccion = np.sum(activaciones_prediccion == 0, axis=1)
ceros_entrenamiento = np.sum(activaciones_entrenamiento == 0, axis=1)

print("\n🔍 Neuronas con activación = 0 (apagadas):")
print("-" * 60)
print(f"{'Muestra':<10} {'Predicción':<20} {'Entrenamiento':<20}")
print("-" * 60)
for i in range(10):
    print(f"{i+1:<10} {ceros_prediccion[i]:<20} {ceros_entrenamiento[i]:<20}")
print("-" * 60)

# Promedios
print(f"\n📈 Promedios:")
print(f"   • Neuronas apagadas en PREDICCIÓN: {ceros_prediccion.mean():.2f}")
print(f"   • Neuronas apagadas en ENTRENAMIENTO: {ceros_entrenamiento.mean():.2f}")

# Tasa de dropout esperada
dropout_rate = 0.4  # Según el modelo
neuronas_totales = activaciones_prediccion.shape[1]
neuronas_esperadas_apagadas = neuronas_totales * dropout_rate

print(f"\n🎯 Dropout configurado: {dropout_rate*100}%")
print(f"   • Neuronas totales: {neuronas_totales}")
print(f"   • Neuronas que DEBERÍAN apagarse: ~{neuronas_esperadas_apagadas:.0f}")
print(f"   • Neuronas que SE APAGAN realmente: ~{ceros_entrenamiento.mean():.0f}")

diferencia_porcentual = abs(ceros_entrenamiento.mean() - neuronas_esperadas_apagadas) / neuronas_esperadas_apagadas * 100
print(f"   • Diferencia: {diferencia_porcentual:.1f}% (cercano = funciona bien)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Comparación de neuronas apagadas
axes[0].bar(['Predicción\n(Dropout OFF)', 'Entrenamiento\n(Dropout ON)'],
            [ceros_prediccion.mean(), ceros_entrenamiento.mean()],
            color=['green', 'coral'], alpha=0.7, edgecolor='black')
axes[0].axhline(neuronas_esperadas_apagadas, color='red', linestyle='--', 
                linewidth=2, label=f'Esperado ({dropout_rate*100}%)')
axes[0].set_ylabel('Neuronas apagadas (promedio)', fontsize=12)
axes[0].set_title('Neuronas Apagadas: Predicción vs Entrenamiento', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gráfico 2: Heatmap de activaciones
# Mostrar solo las primeras 50 neuronas para legibilidad
axes[1].imshow(activaciones_entrenamiento[:, :50].T, cmap='RdYlGn', aspect='auto')
axes[1].set_xlabel('Muestra', fontsize=12)
axes[1].set_ylabel('Neurona (primeras 50)', fontsize=12)
axes[1].set_title('Activaciones en Modo Entrenamiento\n(Rojo = apagada, Verde = activa)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xticks(range(10))
axes[1].set_xticklabels([f'M{i+1}' for i in range(10)])

plt.tight_layout()
plt.show()

print("\n✅ Conclusión del Método 3:")
print("   • En modo PREDICCIÓN: Pocas o ninguna neurona apagada")
print("   • En modo ENTRENAMIENTO: ~40% de neuronas apagadas (coincide con dropout=0.4)")
print("   • Esto confirma que el Dropout está funcionando correctamente")

### 📚 **Resumen: Cómo Funciona el Dropout**

#### 🔑 Conceptos Clave

```
┌─────────────────────────────────────────────────────────────┐
│                  DROPOUT: Lo que SÍ y NO hace               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ❌ NO HACE:                                                │
│     • NO pone los PESOS a cero                              │
│     • NO elimina conexiones permanentemente                 │
│     • NO modifica la arquitectura del modelo                │
│                                                             │
│  ✅ SÍ HACE:                                                 │
│     • Apaga ACTIVACIONES (salidas de neuronas) a 0          │
│     • Lo hace ALEATORIAMENTE en cada batch                  │
│     • Solo durante ENTRENAMIENTO (training=True)            │
│     • En predicción usa TODAS las neuronas (escaladas)      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

#### 🧮 Matemática del Dropout

Durante **ENTRENAMIENTO** con dropout rate `p = 0.4`:

```
activación_original = f(W·x + b)    # Salida normal de la neurona

máscara = random_binary(p=0.4)       # 40% de probabilidad de ser 0
activación_dropout = activación_original * máscara / (1-p)

# Ejemplo con 5 neuronas:
activación_original = [0.5, 0.8, 0.3, 0.9, 0.2]
máscara = [1, 0, 1, 0, 1]           # Aleatoria
activación_dropout = [0.83, 0, 0.5, 0, 0.33]  # Escalada por 1/(1-0.4)
```

Durante **PREDICCIÓN**:
```
activación_predicción = f(W·x + b)   # Usa todas las neuronas, sin máscara
                                      # (Ya está escalada por el entrenamiento)
```

#### 🎯 Tres Métodos de Verificación

| Método | Qué Verifica | Resultado Esperado |
|--------|--------------|-------------------|
| **1. Inspección de pesos** | Los pesos NO son cero | Distribución normal de pesos, pocos o ningún cero exacto |
| **2. Predicciones repetidas** | Variabilidad en training mode | Predicciones varían en `training=True`, son estables en `training=False` |
| **3. Activaciones intermedias** | Neuronas apagadas en training | ~p% de activaciones = 0 en training mode, casi 0% en predicción |

#### 💡 Por Qué Funciona el Dropout

1. **Evita co-adaptación**: Las neuronas no pueden depender de neuronas específicas
2. **Ensemble implícito**: Cada batch entrena una "sub-red" diferente
3. **Regularización**: Fuerza a la red a aprender representaciones robustas
4. **Generalización**: El modelo aprende múltiples caminos para la misma predicción

---

**🔬 Experimento adicional**: Prueba entrenar dos modelos idénticos, uno con Dropout y otro sin él, y compara las curvas de validación. El modelo con Dropout debería tener menor gap entre train y validation accuracy (menos overfitting).

## 1️⃣3️⃣ Experimentación y Mejoras

### 🔬 Experimentos Sugeridos:

1. **Cambiar la arquitectura:**
   - Añade más capas ocultas
   - Prueba con diferentes números de neuronas (256, 512)
   - Experimenta con diferentes tasas de Dropout (0.1, 0.4, 0.5)

2. **Hiperparámetros:**
   - Prueba diferentes learning rates
   - Cambia el batch size (128, 512, 1024)
   - Aumenta o reduce epochs

3. **Técnicas de balanceo:**
   - SMOTE (oversampling sintético)
   - Undersampling de la clase mayoritaria
   - Diferentes pesos de clase

4. **Feature Engineering:**
   - Crea interacciones entre features (ej: Temp * Humidity)
   - Extrae mes/estación del año de la fecha
   - Agrupa ubicaciones por regiones

5. **Diferentes activaciones:**
   - Prueba LeakyReLU, ELU, SELU
   - Experimenta con BatchNormalization

6. **Regularización:**
   - Añade regularización L1/L2 a las capas Dense
   - Implementa Early Stopping

7. **Ensemble:**
   - Entrena múltiples modelos y promedia predicciones
   - Combina con otros algoritmos (Random Forest, XGBoost)

1. Modificaciones en la Arquitectura
    - Añadir más capas ocultas
        * Razonamiento: El dataset es grande (~145k muestras) con patrones meteorológicos complejos (interacciones no lineales entre presión, temperatura, humedad). Capas adicionales permiten capturar jerarquías abstractas (ej: capa 1 detecta patrones locales, capa 2 patrones regionales, capa 3+ interacciones climáticas globales).
        * Trade-off: Cada capa nueva aumenta parámetros → riesgo de sobreajuste si no aumentas regularización. Prueba añadir 1-2 capas más (ej: 128→64→32→16→8).
        * Práctico: Comienza con 4 capas totales y monitoriza validation loss. Si empieza a aumentar antes que training loss, reduce complejidad.
    - Más neuronas (256, 512)
        * Razonamiento: La lluvia depende de interacciones complejas (múltiples variables atmosféricas). Más neuronas = mayor capacidad de representación para capturar relaciones de alta dimensionalidad.
        * Problema: Con 23 features, 512 neuronas puede ser excesivo (ratio 22:1). Esto puede hacer al modelo "recordar" en lugar de "generalizar".
        * Recomendación: Deveriamos probar expansiones progresivas: 128→256→128→64 (botleneck architecture) o 256→128→64→32.
    - Dropout variables (0.1, 0.4, 0.5)
        * Razonamiento: Dropout=0.3 es conservador. Tasas más altas (0.4-0.5) fuerzan robustez en redes muy profundas, pero pueden "matar" demasiada información. Tasas bajas (0.1) son útiles en capas iniciales donde cada feature es valiosa.
        * Estrategia inteligente: Dropout decreciente → 0.4 en capa 1 (mucha regularización), 0.3 en capa 2, 0.2 en capa 3. O dropout espacial en features correlacionadas.
2. Hiperparámetros
    - Learning rates
        * Impacto: Un LR bajo (0.0001) converge lento pero preciso; alto (0.01) aprende rápido pero puede diverger. Tu problema es imbalanced (más días sin lluvia), por lo que un LR adaptativo es crucial.
        * Técnica recomendada: Learning Rate Scheduling (ReduceLROnPlateau). Empieza con 0.001, reduce un 50% cuando validation loss estancado. O prueba AdamW con weight decay.
    - Batch size (128, 512, 1024)
        * Razonamiento: Batch size afecta directamente la varianza del gradiente. En problemas meteorológicos con datos temporales, batches grandes (1024) capturan mejor la distribución global pero pierden detalles locales. Batches pequeños (64-128) son ruidosos pero generalizan mejor.
        * Trade-off: GPU memory vs. generalización. Prueba ciclos de batch size: empezar con 256 (estable), aumentar a 512 (acelerar), disminuir a 128 (refinar).
    - Épocas
        * Clave: Con Early Stopping, no necesitas preocuparte por epochs. Establece epochs=100 con patience=10 epochs. Tu dataset grande necesita al menos 30-50 epochs para convergencia.
3. Técnicas de Balanceo
    - El problema: En Australia, ~80% de los días NO llueve. Tu Precision=0.5775 es bajo porque el modelo es conservador (prefiere predecir "No llueve").
    - SMOTE (Synthetic Minority Oversampling)
        * Razonamiento: Genera muestras sintéticas de días de lluvia combinando features cercanas. Útil porque crea variedad en la clase minoritaria sin duplicar información.
        * Riesgo: En datos temporales, SMOTE puede romper dependencias secuenciales. Solución: Aplicar SMOTE solo después de separar train/test, y solo en training set. Usa SMOTENC si hay categóricas.
    - Undersampling mayoritaria
        * Razonamiento: Elimina aleatoriamente días sin lluvia hasta balancear (50/50). Rápido y efectivo, pero pierdes información valiosa de "días normales".
        * Cuándo usar: Si tienes memoria limitada o SMOTE genera overfitting. Mezcla con ENN (Edited Nearest Neighbors) para eliminar ruido.
    - Pesos de clase
        * Implementación: class_weight={0:1, 1:3} (penaliza 3× más el error en lluvia). Es la técnica más limpia ya que no modifica datos.
        * Cálculo: weight = total_samples/(n_classes * n_samples_class). Para 80/20, peso de lluvia ≈ 4.0.
        * Ventaja: El modelo aprende a ser más sensible a lluvia sin complejidad adicional.
4. Feature Engineering Crítico
    - Interacciones features: Temp * Humidity, Pressure * WindSpeed
        * Razonamiento: La lluvia no depende de variables aisladas. Humedad alta + temperatura baja = alta probabilidad de precipitación. Multiplicaciones capturan no-linearidades implícitas.
        * Práctico: Usa PolynomialFeatures(degree=2) o crea interacciones manuales climáticamente válidas (ej: DewPoint = Temp - (100-Humidity)/5).
    - Extracción temporal
        * Impacto: La lluvia en Australia es estacional (monzones en norte, fríos en sur). Extraer Month, Season (0:winter, 1:spring...) añade periodicidad cíclica.
        * Encoding: No uses month como 1-12 (implica orden). Usa sin/cos encoding: month_sin = sin(2π*month/12), month_cos = cos(2π*month/12).
    - Agrupación regional
        * Razonamiento: 23 estaciones con latitudes/longitudes diferentes. Agrupa por Region (North, South, East, West, Central) permite al modelo aprender microclimas.
        * Complejidad: Añade embedding layers si creas muchas regiones. O usa coordenadas como features numéricas con transformación de distancia.
5. Activaciones Avanzadas
    - LeakyReLU/ELU
        * Problema de ReLU: "Muerte de neuronas" (gradiente 0 para entradas negativas). En meteorología, muchas relaciones son unidireccionales (ej: presión siempre positiva).
        * LeakyReLU (α=0.01): Mantiene gradiente mínimo → neuronas no mueren. ELU: Suave en negativo → convergencia más rápida en épocas iniciales.
        * Recomendación: Prueba LeakyReLU en capas intermedias, especialmente si ves muchas neuronas con peso=0.
    - SELU + AlphaDropout
        * Auto-normalización: SELU mantiene media/varianza constantes. Útil para redes muy profundas (>5 capas).
        * AlphaDropout: Mantiene propiedades de SELU. Si añades capas, reemplaza Dropout normal por AlphaDropout.
    - BatchNormalization
        * Razonamiento: Normaliza activaciones por batch. Acelera convergencia 10× y permite LR más altos. Crucial con activaciones como SELU.
        * Posición: Después de capa densa, antes de activación (o después, ambas funcionan). En tu caso: Dense → BatchNorm → ReLU → Dropout.
        * Ventaja: Reduce necesidad de Dropout (puedes bajar a 0.2).
6. Regularización
    - L1/L2 en Dense layers
        * L2 (λ=0.01): Penaliza pesos grandes → modelo más "simple". Reduce sobreajuste en datasets ruidosos (datos meteorológicos tienen mediciones erróneas).
        * L1: Encourages sparse weights (selección de features). Útil si sospechas que algunas features son irrelevantes.
        * Implementación: Dense(128, activation='relu', kernel_regularizer=l2(0.01)).
    - Early Stopping
        * Configuración: monitor='val_loss', patience=15, restore_best_weights=True. Detiene si no mejora en 15 epochs.
        * Crucial: Con 145k muestras, el modelo puede sobreajustar después de 40-50 epochs. Early Stopping evita entrenamiento innecesario.
    - Dropout + L2
        * Sinergia: El efecto es multiplicativo. Usa L2 bajo (0.001) con Dropout moderado (0.2-0.3). No abuses o el modelo subajusta.
7. Ensemble (Máxima Ganancia)
    - Bagging de redes neuronales
        * Razonamiento: Entrena 5 modelos con diferentes inicializaciones o subset de datos. Promedia predicciones → reduce varianza. En meteorología, esto suaviza predicciones erráticas.
        * Implementación: keras.utils.timeseries_dataset_from_array con diferentes seed. O usa BaggingClassifier de sklearn con KerasClassifier.
    - Stacking con XGBoost/Random Forest
        * Sinergia: Red neuronal captura interacciones complejas; XGBoost captura interacciones de orden bajo y es robusto a outliers. Combinarlos mejora robustez.
        * Pipeline:
            1. Entrena red neuronal (predict_proba)
            2. Entrena XGBoost en mismos datos
            3. Meta-modelo (logística) con outputs de ambos como features
    - Snapshot Ensemble
        * Eficiente: Un solo entrenamiento guarda checkpoints cada 10 epochs con ciclos de LR. Combina 5 "instantáneas" del modelo. Rápido y efectivo.
    - Plan de Acción Recomendado
        * Inmediato: Añade class_weight y EarlyStopping → subir Recall a >0.85
        * Feature Engineering: Month encoding + interacciones Temp×Humidity → Accuracy +2-3%
        * Arquitectura: Añade BatchNorm, prueba LeakyReLU → estabilizar training
        * Avanzado: SMOTE + Ensemble → Precision >0.70, AUC >0.92

El AUC=0.8968 es bueno, pero la baja Precision indica que el modelo es reacio a predecir "lluvia". Enfócate primero en balanceo y pesos de clase antes de complejizar la arquitectura.

---

## 📚 **SECCIÓN AVANZADA: Modelo Mejorado con Feature Engineering**

Esta sección ha sido **movida a un notebook independiente** para facilitar su estudio:

👉 **[modelo_mejorado_feature_engineering.ipynb](./modelo_mejorado_feature_engineering.ipynb)**

### 🚀 **En ese notebook encontrarás:**

- Feature Engineering Avanzado (10+ nuevas características)
- Arquitectura optimizada (BatchNorm + LeakyReLU + L2)
- Balanceo de clases (SMOTE y class weights)
- Callbacks inteligentes (EarlyStopping + ReduceLR)
- Evaluación con doble umbral (0.4 y 0.5)
- Ensemble learning opcional

---

## 📋 Instrucciones de Uso

1. Instala dependencias: pip install tensorflow imblearn scikit-learn matplotlib
2. Reemplaza la sección de datos sintéticos con tu pd.read_csv('rain_australia.csv')
3. Activa/desactiva mejoras comentando/descomentando bloques marcados con ⚡

🏆 Resultados Esperados

|Métrica	|Baseline	|Con Mejoras	|Ganancia|
|-----------|-----------|---------------|--------|
|Recall	|0.7860	|0.85-0.90	|+8-12%|
|Precision	|0.5775	|0.65-0.70	|+7-12%|
|AUC	|0.8968	|0.92-0.94	|+2-4%|
|Overfitting	|Sí	|No	|✓|

El Recall es la métrica clave: representa cuántos días de lluvia detectamos. Un Recall=0.85 significa que avisamos correctamente el 85% de los días que lloverán.

---

## ✅ **Correcciones Aplicadas: Adaptación al Dataset Real**

### ✅ **Solución Implementada**

La función `ingenieria_features()` ahora **crea automáticamente** estas columnas como agregaciones inteligentes:

#### **Paso 1: Creación de Promedios**
```python
df['Pressure'] = (df['Pressure9am'] + df['Pressure3pm']) / 2
df['Temp'] = (df['Temp9am'] + df['Temp3pm']) / 2
df['Humidity'] = (df['Humidity9am'] + df['Humidity3pm']) / 2
df['WindSpeed'] = (df['WindSpeed9am'] + df['WindSpeed3pm']) / 2
```

#### **Paso 2: Creación de Deltas (cambios 9am→3pm)**
```python
df['Delta_Pressure'] = df['Pressure3pm'] - df['Pressure9am']
df['Delta_Humidity'] = df['Humidity3pm'] - df['Humidity9am']
df['Delta_WindSpeed'] = df['WindSpeed3pm'] - df['WindSpeed9am']
```

#### **Paso 3: Uso en Interacciones**
Las interacciones ahora funcionan correctamente usando los promedios creados:
```python
df['Presion_Temperatura'] = df['Pressure'] * df['Temp']  # ✅ Ahora funciona
df['Humedad_Viento'] = df['Humidity'] * df['WindSpeed']  # ✅ Ahora funciona
```

### 📊 **Mapeo de Columnas**

| Columna Original (Dataset) | Columna Creada | Tipo |
|---------------------------|----------------|------|
| `Pressure9am`, `Pressure3pm` | `Pressure` | Promedio |
| `Pressure9am`, `Pressure3pm` | `Delta_Pressure` | Diferencia (evolución) |
| `Temp9am`, `Temp3pm` | `Temp` | Promedio |
| `Humidity9am`, `Humidity3pm` | `Humidity` | Promedio |
| `Humidity9am`, `Humidity3pm` | `Delta_Humidity` | Diferencia |
| `WindSpeed9am`, `WindSpeed3pm` | `WindSpeed` | Promedio |
| `WindSpeed9am`, `WindSpeed3pm` | `Delta_WindSpeed` | Diferencia |
| `MaxTemp`, `MinTemp` | `Delta_Temp` | Amplitud térmica |

### 🌍 **Agrupación Geográfica Ampliada**

Se añadió mapeo completo para las **49 locaciones** del dataset, agrupadas en 5 regiones:
- **North** (4 ciudades): Darwin, Cairns, Townsville, Katherine
- **South** (12 ciudades): Melbourne, Hobart, Adelaide, etc.
- **East** (16 ciudades): Sydney, Brisbane, Canberra, etc.
- **West** (6 ciudades): Perth, Albany, PerthAirport, etc.
- **Central** (6 ciudades): Uluru, AliceSprings, Cobar, etc.

---

---

## 🔥 **Siguiente Nivel: Ensemble Stacking**

¿Quieres llevar tu modelo al máximo nivel? 

🚀 **Consulta el notebook especializado**: `ensemble_stacking_rain_australia.ipynb`

Este notebook independiente implementa un **Ensemble Stacking** que combina:
- 🧠 **Red Neuronal Profunda**
- 🌳 **XGBoost** (Gradient Boosting)
- 🌲 **Random Forest**
- 🎯 **Meta-modelo** (Regresión Logística)

### 📈 Ventajas del Ensemble:
- ✅ AUC mejorado en +3-4% (de ~89% a ~92-93%)
- ✅ Mayor Recall (detecta más días de lluvia)
- ✅ Menor varianza (predicciones más estables)
- ✅ Robustez ante datos cambiantes

---